# Full Fine-tuning with SmolLM2 135M using Unsloth

This notebook demonstrates full fine-tuning (not LoRA) of the SmolLM2 135M parameter model using Unsloth. Full fine-tuning updates all model parameters rather than just a small subset, which can lead to better performance but requires more memory and compute.

## What we'll cover:
- Installing Unsloth and dependencies
- Loading the SmolLM2 135M model with full fine-tuning enabled
- Preparing a dataset for instruction fine-tuning
- Training the model
- Testing inference with the fine-tuned model

## About SmolLM2 135M:
SmolLM2 is a small language model designed to be efficient while maintaining good performance. The 135M parameter version is ideal for quick experimentation and can be fully fine-tuned on consumer hardware.

In [1]:
# Install Unsloth and dependencies - optimized for Colab
# This version uses pre-built wheels to avoid long compilation times

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-4fiajxnx/unsloth_f0775788b8634174a2c1f21882e561b9
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-4fiajxnx/unsloth_f0775788b8634174a2c1f21882e561b9
  Resolved https://github.com/unslothai/unsloth.git to commit 341ce85864d191e4a6b7c447b9167c1faf5e20d3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.5/283.5 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 152.9 MB/s eta 0:00:00

## Import Required Libraries

We'll import the necessary libraries for fine-tuning:
- `unsloth`: For optimized model loading and training
- `transformers`: For tokenizer and model utilities
- `trl`: For supervised fine-tuning trainer
- `torch`: For PyTorch operations

We'll also verify that GPU is available and check memory usage.

In [1]:
# Import necessary libraries
from unsloth import FastLanguageModel
import torch
from transformers import TrainingArguments
from trl import SFTTrainer
from datasets import load_dataset

# Check if GPU is available
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU Available: True
GPU Name: NVIDIA L4
GPU Memory: 23.80 GB


## Model Configuration for Full Fine-tuning

For full fine-tuning, we need to load the model without quantization since all parameters need to be trainable.

- `max_seq_length`: Maximum sequence length for training (2048 tokens)
- `dtype`: Data type for computation (None means auto-detection)
- `load_in_4bit`: Set to False for full fine-tuning (quantized models can't be fully fine-tuned)
- `model_name`: The specific model to use from HuggingFace

Note: Full fine-tuning without quantization requires more GPU memory but updates all model parameters.

In [2]:
# Configuration parameters for full fine-tuning
max_seq_length = 2048  # Maximum sequence length
dtype = None  # Auto-detect dtype (will use float16 for efficiency)
load_in_4bit = False  # Must be False for full fine-tuning

# Model selection - using SmolLM2 135M
model_name = "unsloth/SmolLM2-135M-Instruct"

print(f"Model: {model_name}")
print(f"Max Sequence Length: {max_seq_length}")
print(f"4-bit Quantization: {load_in_4bit}")
print("Full fine-tuning enabled (no quantization)")

Model: unsloth/SmolLM2-135M-Instruct
Max Sequence Length: 2048
4-bit Quantization: False
Full fine-tuning enabled (no quantization)


## Load Model and Tokenizer for Full Fine-tuning

We load the model without quantization to enable full fine-tuning of all parameters.

In [3]:
# Load model and tokenizer for full fine-tuning
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,  # False for full fine-tuning
    full_finetuning=True,  # THIS IS THE KEY PARAMETER for full fine-tuning
)

print("Model loaded successfully for full fine-tuning!")
print(f"Model type: {type(model)}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

==((====))==  Unsloth 2025.11.3: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using bfloat16 full finetuning which cuts memory usage by 50%.
To enable float32 training, use `float32_mixed_precision = True` during FastLanguageModel.from_pretrained


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/423 [00:00<?, ?B/s]

Model loaded successfully for full fine-tuning!
Model type: <class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>
Total parameters: 134,515,584
Trainable parameters: 134,515,584


## Load Training Dataset

We'll use a dataset for instruction fine-tuning. For this example, we'll use the "mlabonne/guanaco-llama2-1k" dataset, which contains 1000 high-quality instruction-response pairs.

The dataset format typically includes:
- `text`: The full conversation in a chat template format

We'll format the data according to the model's chat template to ensure proper training.

In [4]:
# Load dataset from HuggingFace
dataset = load_dataset("mlabonne/guanaco-llama2-1k", split="train")

print(f"Dataset loaded successfully!")
print(f"Number of examples: {len(dataset)}")
print(f"Dataset columns: {dataset.column_names}")
print("\nFirst example:")
print(dataset[0])

Dataset loaded successfully!
Number of examples: 1000
Dataset columns: ['text']

First example:
{'text': '<s>[INST] Me gradué hace poco de la carrera de medicina ¿Me podrías aconsejar para conseguir rápidamente un puesto de trabajo? [/INST] Esto vale tanto para médicos como para cualquier otra profesión tras finalizar los estudios aniversarios y mi consejo sería preguntar a cuántas personas haya conocido mejor. En este caso, mi primera opción sería hablar con otros profesionales médicos, echar currículos en hospitales y cualquier centro de salud. En paralelo, trabajaría por mejorar mi marca personal como médico mediante un blog o formas digitales de comunicación como los vídeos. Y, para mejorar las posibilidades de encontrar trabajo, también participaría en congresos y encuentros para conseguir más contactos. Y, además de todo lo anterior, seguiría estudiando para presentarme a las oposiciones y ejercer la medicina en el sector público de mi país. </s>'}


## Format Dataset for Training

We need to format our dataset according to the model's chat template. This ensures the model learns the proper instruction-following format.

The chat template typically follows this structure:
- System message (optional)
- User message (instruction)
- Assistant message (response)

We'll create a formatting function that converts our dataset into the proper format using the tokenizer's chat template.

In [5]:
# Define the chat template formatting function
def formatting_prompts_func(examples):
    """
    Format the dataset examples into chat template format.
    Handles both single examples and batches.
    """
    texts = []
    for text in examples["text"]:
        # The dataset already contains formatted text
        # We just need to ensure it's in the right format
        texts.append(text)
    return {"text": texts}

# Apply formatting to dataset
formatted_dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
)

print("Dataset formatted successfully!")
print("\nFormatted example:")
print(formatted_dataset[0]["text"][:500])  # Print first 500 characters

Dataset formatted successfully!

Formatted example:
<s>[INST] Me gradué hace poco de la carrera de medicina ¿Me podrías aconsejar para conseguir rápidamente un puesto de trabajo? [/INST] Esto vale tanto para médicos como para cualquier otra profesión tras finalizar los estudios aniversarios y mi consejo sería preguntar a cuántas personas haya conocido mejor. En este caso, mi primera opción sería hablar con otros profesionales médicos, echar currículos en hospitales y cualquier centro de salud. En paralelo, trabajaría por mejorar mi marca personal


## Setup Training Configuration

We configure the training parameters using HuggingFace's `TrainingArguments`. These parameters control how the model is trained:

Key parameters:
- `per_device_train_batch_size`: Number of samples per batch (2 for memory efficiency)
- `gradient_accumulation_steps`: Accumulate gradients over multiple steps (4 steps = effective batch size of 8)
- `warmup_steps`: Number of steps for learning rate warmup (5 steps)
- `max_steps`: Total training steps (60 steps for quick demo)
- `learning_rate`: Learning rate for optimization (2e-4)
- `fp16/bf16`: Mixed precision training for speed
- `logging_steps`: How often to log metrics (1 step)
- `optim`: Optimizer type (adamw_8bit for memory efficiency)
- `weight_decay`: L2 regularization (0.01)
- `lr_scheduler_type`: Learning rate schedule (linear decay)
- `seed`: Random seed for reproducibility (3407)
- `output_dir`: Where to save model checkpoints
- `gradient_checkpointing`: Disabled to avoid compatibility issues

In [6]:
# Setup training arguments for full fine-tuning
training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=60,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="outputs",
    report_to="none",
    gradient_checkpointing=False,  # Disable to avoid compatibility issues
)

print("Training arguments configured!")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Total training steps: {training_args.max_steps}")
print(f"Learning rate: {training_args.learning_rate}")

Training arguments configured!
Effective batch size: 8
Total training steps: 60
Learning rate: 0.0002


## Initialize the Supervised Fine-tuning Trainer

We use TRL's `SFTTrainer` (Supervised Fine-Tuning Trainer) which is optimized for instruction fine-tuning tasks.

The trainer handles:
- Loading and batching data
- Forward and backward passes
- Gradient accumulation
- Optimizer updates
- Logging and checkpointing

Key parameters:
- `model`: The model to train
- `tokenizer`: Tokenizer for processing text
- `train_dataset`: Training data
- `dataset_text_field`: Which column contains the text (default is "text")
- `max_seq_length`: Maximum sequence length for truncation
- `dataset_num_proc`: Number of processes for data loading (2 for parallel processing)
- `packing`: Whether to pack multiple examples into one sequence (False for clarity)
- `args`: Training arguments we configured earlier

In [7]:
# Initialize the SFTTrainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

print("Trainer initialized successfully!")
print(f"Training dataset size: {len(trainer.train_dataset)}")

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1000 [00:00<?, ? examples/s]

Trainer initialized successfully!
Training dataset size: 1000


## Train the Model

Now we start the training process. The trainer will:
- Iterate through the dataset for the specified number of steps
- Compute loss and gradients
- Update model parameters
- Log training metrics (loss, learning rate, etc.)

This will take several minutes depending on your GPU. You'll see the training progress with loss values and other metrics.

For full fine-tuning, all 135 million parameters are being updated, which is why this takes more time and memory compared to LoRA fine-tuning.

In [8]:
# Start training
print("Starting training...")
trainer_stats = trainer.train()

print("\nTraining completed!")
print(f"Training loss: {trainer_stats.training_loss:.4f}")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.2f} seconds")
print(f"Training samples per second: {trainer_stats.metrics['train_samples_per_second']:.2f}")

The model is already on multiple devices. Skipping the move to device specified in `args`.


Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 134,515,584 of 134,515,584 (100.00% trained)


Step,Training Loss
1,2.360700
2,2.651200
3,2.255200
4,2.051000
5,2.493800
6,2.146200
7,2.454600
8,2.521000
9,2.126500
10,2.423300


Unsloth: Will smartly offload gradients to save VRAM!

Training completed!
Training loss: 2.0425
Training time: 73.93 seconds
Training samples per second: 6.49


## Test the Fine-tuned Model with Inference

After training is complete, we'll test the model by generating responses to sample prompts.

We need to:
1. Switch the model to inference mode using `FastLanguageModel.for_inference()`
2. Prepare a test prompt in the same format used during training
3. Tokenize the input
4. Generate a response
5. Decode and display the output

This will help us see if the model has learned from the training data.

In [9]:
# Enable fast inference mode
FastLanguageModel.for_inference(model)

# Create a test prompt
test_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is the capital of France?

### Response:
"""

# Tokenize the input
inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")

# Generate response
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

print("Model response:")
outputs = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=128,
    use_cache=True,
    temperature=0.7,
    top_p=0.9
)

Model response:
The capital of France is Paris.

### Explanation:
The capital of France is Paris, which is located in the northern part of the country. Paris is known for its historical landmarks, cultural institutions, and cultural events. It is also the largest city in France and the third-largest city in Europe by population.

### Conclusion:
Paris is a city that is famous for its historical landmarks, cultural institutions, and cultural events, making it a popular destination for tourists and visitors.<|im_end|>


## Save the Fine-tuned Model

Now we'll save the fine-tuned model so it can be used later. Unsloth provides several saving options:

1. **save_pretrained**: Saves the full model in standard HuggingFace format
2. **save_pretrained_merged**: Saves the model with weights merged (for full fine-tuning, this is the standard approach)
3. **push_to_hub_merged**: Upload directly to HuggingFace Hub

For full fine-tuning, we save the entire model since all parameters have been updated. We'll save in 16-bit format for a good balance between quality and file size.

In [11]:
# Save the model locally
model_save_path = "smollm2_135m_full_finetuned"

# For full fine-tuning, save the merged model
model.save_pretrained_merged(
    model_save_path,
    tokenizer,
    save_method="merged_16bit",  # Can also use "merged_4bit" for smaller size
)

print(f"Model saved successfully to: {model_save_path}")
print(f"You can load this model later using FastLanguageModel.from_pretrained('{model_save_path}')")

/usr/local/lib/python3.12/dist-packages/unsloth_zoo/saving_utils.py:969: UserWarning: Model is not a PeftModel (no Lora adapters detected). Skipping Merge. Please use save_pretrained() or push_to_hub() instead!
  warnings.warn("Model is not a PeftModel (no Lora adapters detected). Skipping Merge. Please use save_pretrained() or push_to_hub() instead!")


Model saved successfully to: smollm2_135m_full_finetuned
You can load this model later using FastLanguageModel.from_pretrained('smollm2_135m_full_finetuned')
